# Lektion 18: Sicherung von KI-Agenten mit kryptografischen Belegen

## Praxis-Notizbuch

Dieses Notizbuch führt durch vier Übungen:

1. **Unterschreiben Sie Ihren ersten Beleg** für einen Agenten-Tool-Aufruf und verifizieren Sie ihn.
2. **Manipulieren Sie den Beleg** und beobachten Sie, wie die Verifikation fehlschlägt.
3. **Bauen Sie eine Kette mit drei Belegen auf** und prüfen Sie die Integrität der Kette.
4. **Verpacken Sie einen Microsoft Agent Framework Tool-Aufruf** so, dass jede Aktion einen Beleg ausgibt.

Alle kryptografischen Primitive werden aus gut gepflegten Bibliotheken importiert (`pynacl` für Ed25519, `jcs` für RFC 8785 kanonisches JSON, `hashlib` aus der Python-Standardbibliothek für SHA-256). Die Beleg-Logik selbst ist reines Python, das Sie lesen und ändern können.

Führen Sie die Zellen der Reihe nach aus. Jeder Abschnitt ist kurz und in sich abgeschlossen.


## Einrichtung

Installieren Sie die beiden Abhängigkeiten. Beide haben großzügige Lizenzen (Apache-2.0 / MIT).


In [1]:
%pip install -q pynacl jcs

Note: you may need to restart the kernel to use updated packages.


In [2]:
import json
import hashlib
import base64
from datetime import datetime, timezone

from nacl import signing
from nacl.exceptions import CryptoError
from jcs import canonicalize

## Hilfsprogramme

Diese beiden Hilfsprogramme kümmern sich um die base64url-Codierung (ohne Padding) und das SHA-256-Hashing beliebiger Objekte. Sie halten den Rest des Notebooks auf die eigentliche Quittungslogik fokussiert.


In [3]:
def b64url_nopad(data: bytes) -> str:
    """Base64url-encode bytes without padding (RFC 4648 Section 5)."""
    return base64.urlsafe_b64encode(data).decode("ascii").rstrip("=")

def b64url_decode(s: str) -> bytes:
    """
    Decode unpadded base64url strictly, raising ValueError for anything else.

    Python's decoder skips characters outside the alphabet and also accepts
    padding, the standard +/ alphabet and non-zero spare bits, so the same key
    or signature could be written several ways. Re-encoding and comparing
    accepts only the one form that b64url_nopad produces.
    """
    padding = "=" * ((4 - len(s) % 4) % 4)
    data = base64.urlsafe_b64decode(s + padding)
    if b64url_nopad(data) != s:
        raise ValueError("not canonical unpadded base64url")
    return data

def sha256_canonical(obj) -> str:
    """
    SHA-256 hash of a Python object, computed over its JCS-canonical JSON form.
    Returns a 'sha256:' prefixed hex digest so callers can identify the algorithm.
    """
    canonical = canonicalize(obj)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

## Abschnitt 1: Signieren Sie Ihre erste Quittung

Stellen Sie sich vor, unser Agent für **Contoso Travel** hat gerade Flüge von Sydney nach Los Angeles für einen Kunden gesucht. Wir möchten die Aussage des Gateways über diesen Tool-Aufruf aufzeichnen, damit ein zukünftiger Prüfer sie unter einem separat angehefteten Aussteller-Schlüssel überprüfen kann.

### Schritt 1.1: Erstellen Sie einen Signaturschlüssel

In der Produktion würde der Signaturschlüssel des Agents in einem Hardware-Sicherheitsmodul (HSM), Azure Key Vault oder einem ähnlichen geschützten Speicher aufbewahrt. Für diese Lektion generieren wir einen frischen Schlüssel im Arbeitsspeicher.


In [4]:
signing_key = signing.SigningKey.generate()
verify_key = signing_key.verify_key

public_key_b64 = b64url_nopad(bytes(verify_key))
# Trust configuration is out of band. Never trust a receipt only because it embeds a key.
TRUSTED_PUBLIC_KEYS = {public_key_b64}
print(f"Public key (Ed25519, 32 bytes): {public_key_b64}")

Public key (Ed25519, 32 bytes): NKiSqaTHCofGAokp-IyN_y6xJH14J5aUTkL64oMK5uM


### Schritt 1.2: Aufbau der Beleg-Nutzlast

Die Nutzlast enthält alles, was der Beleg bestätigen soll: wer gehandelt hat, mit welchem Werkzeug, mit welchen Argumenten, was zurückkam, unter welcher Richtlinie und wann. Wir hashen die Argumente und das Ergebnis, anstatt sie inline einzufügen, damit der Beleg keine sensiblen Inhalte preisgibt.


In [5]:
tool_args = {
    "origin": "SYD",
    "destination": "LAX",
    "departure_date": "2026-06-15",
    "passengers": 2,
}

tool_result = [
    {"flight": "QF11", "price": 1850, "stops": 0},
    {"flight": "UA864", "price": 1620, "stops": 1},
    {"flight": "DL11", "price": 1740, "stops": 0},
]

payload = {
    "type": "agent.tool_call.v1",
    "agent_id": "contoso-travel-bot",
    "tool_name": "lookup_flights",
    "tool_args_hash": sha256_canonical(tool_args),
    "result_hash": sha256_canonical(tool_result),
    "policy_id": "contoso-travel-policy-v3",
    "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
    "sequence": 0,
    "previous_receipt_hash": None,
}

print(json.dumps(payload, indent=2))

{
  "type": "agent.tool_call.v1",
  "agent_id": "contoso-travel-bot",
  "tool_name": "lookup_flights",
  "tool_args_hash": "sha256:47578acca4df262c8f172b91493b818a26d042e7beb8e7b121e2bc3776152746",
  "result_hash": "sha256:556447bf01c3c33285086d224b3d6fb4cd7b620b5151fbbebe67e7ff81cdde7a",
  "policy_id": "contoso-travel-policy-v3",
  "timestamp": "2026-10-16T04:44:07Z",
  "sequence": 0,
  "previous_receipt_hash": null
}


### Schritt 1.3: Beleg unterschreiben und zusammenstellen

Drei Schritte:

1. Die Nutzlast mit JCS kanonisieren, sodass zwei Implementierungen, die den gleichen logischen Beleg erzeugen, byte-identische Bytes produzieren.
2. Die kanonisierten Bytes direkt mit dem Ed25519-Privatschlüssel signieren. PureEdDSA hasht die Nachricht intern, daher würde ein zusätzliches Pre-Hashing das Protokoll ändern.

Die Signatur wird dann an die ursprüngliche Nutzlast angehängt, um den endgültigen Beleg zu erzeugen.


In [6]:
def sign_receipt(payload: dict, signing_key: signing.SigningKey, verify_key) -> dict:
    """
    Sign a receipt payload. Returns the receipt with attached signature and public key.
    The 'signature' and 'public_key' fields are NOT part of the canonical signed bytes.
    """
    canonical = canonicalize(payload)
    signature_bytes = signing_key.sign(canonical).signature
    return {
        **payload,
        "signature": {
            "alg": "EdDSA",
            "sig": b64url_nopad(signature_bytes),
            "public_key": b64url_nopad(bytes(verify_key)),
        },
    }

receipt = sign_receipt(payload, signing_key, verify_key)
print(json.dumps(receipt, indent=2))

{
  "type": "agent.tool_call.v1",
  "agent_id": "contoso-travel-bot",
  "tool_name": "lookup_flights",
  "tool_args_hash": "sha256:47578acca4df262c8f172b91493b818a26d042e7beb8e7b121e2bc3776152746",
  "result_hash": "sha256:556447bf01c3c33285086d224b3d6fb4cd7b620b5151fbbebe67e7ff81cdde7a",
  "policy_id": "contoso-travel-policy-v3",
  "timestamp": "2026-10-16T04:44:07Z",
  "sequence": 0,
  "previous_receipt_hash": null,
  "signature": {
    "alg": "EdDSA",
    "sig": "NnH0T6lpjVThg1RS29rEu3OG1FptWbQgX18Yt1JAPjNnTyje2Luozt5_0gcdxddQXoS6eH6Vlg5MTcKLrvWnBQ",
    "public_key": "NKiSqaTHCofGAokp-IyN_y6xJH14J5aUTkL64oMK5uM"
  }
}


### Schritt 1.4: Überprüfen des Belegs

Die Überprüfung kehrt den Prozess um. Wir entfernen die Signatur, berechnen die kanonischen Bytes neu und prüfen die Signatur anhand des öffentlichen Schlüssels des Ausstellers.

Die Überprüfung erfolgt offline: kein Dienst muss aufgerufen werden und kein Netzwerkzugang ist erforderlich. Allerdings wird neben dem Beleg noch eine Sache benötigt: der öffentliche Schlüssel des Ausstellers, der über einen Kanal bezogen wird, dem der Prüfer bereits vertraut, wie z. B. einen vom Aussteller veröffentlichten JWK Set oder einen zuvor festgelegten Wert. Das `public_key` Feld innerhalb des Belegs kann diese Rolle nicht übernehmen, da wer auch immer den Beleg schreibt, auch dieses Feld schreibt. Abschnitt 2 zeigt die Fälschung, die dadurch verhindert wird.


In [7]:
def verify_receipt(receipt: dict, trusted_public_key: str) -> bool:
    """
    Verify a receipt's Ed25519 signature against a key you already trust.

    trusted_public_key is the issuer's key, obtained from somewhere other than
    this receipt: a JWK Set, a pinned config value, a key registry. Reading the
    key out of receipt["signature"]["public_key"] instead would only prove the
    receipt is consistent with itself. Section 2 shows exactly how that fails.

    It fails closed: anything malformed returns False instead of raising, so
    one bad receipt cannot stop a batch of them from being checked.
    """
    if not isinstance(receipt, dict) or not isinstance(trusted_public_key, str):
        return False
    sig_obj = receipt.get("signature")
    if not isinstance(sig_obj, dict) or sig_obj.get("alg") != "EdDSA":
        return False
    sig_b64 = sig_obj.get("sig")
    if not isinstance(sig_b64, str):
        return False

    try:
        # Reconstruct the payload that was actually signed (everything except signature).
        payload = {k: v for k, v in receipt.items() if k != "signature"}
        canonical = canonicalize(payload)
        verify_key = signing.VerifyKey(b64url_decode(trusted_public_key))
        verify_key.verify(canonical, b64url_decode(sig_b64))
        return True
    except CryptoError:
        return False
    except Exception:
        # An undecodable key or signature, or a payload that cannot be
        # canonicalized: fail closed rather than raise.
        return False

# public_key_b64 came from Section 1, alongside the signing key. In a real
# deployment it would come from the issuer's JWK Set, not from this notebook.
is_valid = verify_receipt(receipt, public_key_b64)
print(f"Receipt is valid: {is_valid}")

# Regression control for the signature scope in draft revision 02. A receipt
# signed over SHA-256(JCS(payload)) is a signature over different bytes and
# must not verify as a direct-JCS Ed25519 receipt.
prehashed_signature = signing_key.sign(hashlib.sha256(canonicalize(payload)).digest()).signature
prehashed_receipt = {
    **receipt,
    "signature": {**receipt["signature"], "sig": b64url_nopad(prehashed_signature)},
}
print(f"Pre-hashed receipt valid: {verify_receipt(prehashed_receipt, public_key_b64)}")

# Malformed input is rejected, not raised.
for malformed in ({"signature": "x"}, {"signature": {"alg": "EdDSA"}}, "not a receipt"):
    assert verify_receipt(malformed, public_key_b64) is False
print("Malformed receipts rejected: True")

Receipt is valid: True
Pre-hashed receipt valid: False
Malformed receipts rejected: True


Sie sollten `Receipt is valid: True`, `Pre-hashed receipt valid: False` und `Malformed receipts rejected: True` sehen. Der positive Fall beweist, dass der direkte-JCS-Signaturpfad funktioniert; die negative Kontrolle macht die Signaturbereichsregel ausführbar, anstatt sie als Prosa zu belassen; die letzte Zeile zeigt, dass fehlerhafte Eingaben `False` zurückgeben, anstatt eine Ausnahme auszulösen.


## Abschnitt 2: Manipulation des Belegs

Der ganze Sinn von Belegen besteht darin, dass sie manipulationssicher sind. Lassen Sie uns das beweisen.

Wir werden genau ein Zeichen des Belegs ändern und beobachten, wie die Verifizierung fehlschlägt.


In [8]:
import copy

tampered = copy.deepcopy(receipt)

# Modify the policy_id field (this is what an attacker might do to claim
# the action was governed by a more permissive policy than was actually used).
original_policy = tampered["policy_id"]
tampered["policy_id"] = "contoso-travel-policy-PERMISSIVE"

print(f"Original policy_id:  {original_policy}")
print(f"Tampered policy_id:  {tampered['policy_id']}")
print()
print(f"Tampered receipt valid? {verify_receipt(tampered, public_key_b64)}")

Original policy_id:  contoso-travel-policy-v3
Tampered policy_id:  contoso-travel-policy-PERMISSIVE

Tampered receipt valid? False


### Was ist gerade passiert?

Als wir `policy_id` geändert haben, änderten sich die kanonischen Bytes. Die Signatur (die über den ursprünglichen kanonischen Bytes lag) stimmt nun nicht mehr überein. Die Überprüfung gibt korrekt `False` zurück.

Es gibt keine Möglichkeit, ein signiertes Feld zu ändern, also irgendetwas außerhalb des `signature` Objekts, und trotzdem eine gültige Verifizierung gegen den öffentlichen Schlüssel des Ausstellers zu erhalten, es sei denn, der Angreifer verfügt über den privaten Schlüssel des Ausstellers. Das `signature` Objekt selbst ist nicht signiert: `sig` ist die Signatur, und `verify_receipt` ignoriert `public_key`, sodass das Ändern dieses Feldes allein nichts ändert. Solange der private Schlüssel in einem Schlüsseltresor liegt und Prüfer den vom Aussteller unabhängig veröffentlichten Schlüssel verwenden, ist es unmöglich, eine Änderung an einem signierten Feld zu verbergen. Diese Bedingung ist wichtig: Die nächste Zelle zeigt eine modifizierte Quittung, die perfekt gegen einen anderen Schlüssel verifiziert.

Probieren Sie es selbst: Ändern Sie `tool_name` oder `agent_id` oder `timestamp` in der obigen Zelle und führen Sie sie erneut aus. Jede Änderung an einem signierten Feld führt zu einer ungültigen Quittung.


### Die Manipulation, die die Signatur nicht erkennt

Der oben genannte Angreifer hat nicht neu signiert. Ein vorsichtigerer würde: die Quittung verändern,
sie mit einem von ihm selbst generierten Schlüssel signieren und diesen Schlüssel im
`public_key`-Feld eintragen. Das Ergebnis ist eine Quittung, deren Signatur für ihren Inhalt vollständig echt ist.


Führen Sie dies aus und sehen Sie, welche der beiden Prüfungen es entdeckt.


In [9]:
# The attacker generates their own keypair. Nothing stops them.
attacker_key = signing.SigningKey.generate()

forged_payload = {k: v for k, v in receipt.items() if k != "signature"}
forged_payload["policy_id"] = "contoso-travel-policy-PERMISSIVE"

# Re-signed, with the attacker's key embedded where the issuer's used to be.
forged = sign_receipt(forged_payload, attacker_key, attacker_key.verify_key)

print(f"Issuer's public key : {public_key_b64}")
print(f"Key inside the forgery: {forged['signature']['public_key']}")
print()
print(f"Checked against the key inside it: "
      f"{verify_receipt(forged, forged['signature']['public_key'])}")
print(f"Checked against the issuer's key : "
      f"{verify_receipt(forged, public_key_b64)}")

# Assert the results, so the cell fails if the forgery ever passes as the issuer's.
assert verify_receipt(receipt, public_key_b64) is True                    # genuine, issuer's key
assert verify_receipt(forged, forged["signature"]["public_key"]) is True  # forgery, its own key
assert verify_receipt(forged, public_key_b64) is False                    # forgery, issuer's key

Issuer's public key : NKiSqaTHCofGAokp-IyN_y6xJH14J5aUTkL64oMK5uM
Key inside the forgery: nIXlWGm0syJQysGkn-nRlzFS59HlPCTqeQr32TJoHM4

Checked against the key inside it: True
Checked against the issuer's key : False


Sie sollten `True` und dann `False` sehen. Die drei `assert` Zeilen machen daraus einen
Check: Die Zelle löst einen Fehler aus, wenn die Fälschung jemals gegen den
Schlüssel des Ausstellers verifiziert wird.

Die Fälschung ist intern konsistent. Ihre Signatur passt tatsächlich zu ihrem
Inhalt und dem Schlüssel, den sie trägt, so dass ein Prüfer, der den Schlüssel aus dem
Beleg liest, diesen als gültig meldet. An dem Beleg sieht nichts falsch aus, weil
nichts daran inkonsistent ist.

Das Einzige, was sie aufdeckt, ist die Prüfung gegen einen Schlüssel, den der Angreifer nicht
beeinflussen konnte. Deshalb nimmt `verify_receipt` den Schlüssel als Argument: eine
Funktion mit der Signatur `verify_receipt(receipt)` kann nicht sicher geschrieben werden,
weil der einzige Schlüssel, den sie erreichen könnte, derjenige ist, den ein Angreifer bereitgestellt hat.

Derselbe Beleg wird als `sample_receipts/04_resigned_receipt.json` gespeichert, wenn Sie
ihn außerhalb des Notebooks inspizieren möchten.


## Abschnitt 3: Quittungen verketten

Eine einzelne Quittung schützt eine Aktion. Die meisten Agenten führen viele Aktionen aus. Um die gesamte Sequenz manipulationssicher zu machen, verknüpfen wir jede Quittung mit der vorherigen, indem wir den Hash der vorherigen Quittung im Nutzdatenbereich der neuen Quittung einfügen.

```text
Receipt 0  -->  Receipt 1  -->  Receipt 2
                  |                 |
                  +-- previous_receipt_hash field --+
```

Wenn jemand eine Quittung innerhalb der präsentierten Sequenz entfernt oder neu anordnet, wird die nächste Verknüpfung unterbrochen. Das Erkennen von Verkürzungen oder vom Unterzeichner kontrollierten Suffix-Umformulierungen erfordert außerdem einen fixierten aktuellen Kopf, einen externen Prüfpunkt oder ein authentifiziertes Populationsversprechen.


In [10]:
def receipt_hash(receipt: dict) -> str:
    """
    Compute the chain hash of a complete receipt (including signature).
    This becomes the previous_receipt_hash of the next receipt in the chain.
    """
    canonical = canonicalize(receipt)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

def make_receipt(
    tool_name: str,
    tool_args: dict,
    tool_result,
    sequence: int,
    previous_receipt_hash,
    signing_key,
    verify_key,
    agent_id: str = "contoso-travel-bot",
    policy_id: str = "contoso-travel-policy-v3",
) -> dict:
    """Convenience: build, sign, and return a receipt for one tool call."""
    payload = {
        "type": "agent.tool_call.v1",
        "agent_id": agent_id,
        "tool_name": tool_name,
        "tool_args_hash": sha256_canonical(tool_args),
        "result_hash": sha256_canonical(tool_result),
        "policy_id": policy_id,
        "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
        "sequence": sequence,
        "previous_receipt_hash": previous_receipt_hash,
    }
    return sign_receipt(payload, signing_key, verify_key)

In [11]:
# Build a chain of three receipts: search, hold, book.
r0 = make_receipt(
    tool_name="lookup_flights",
    tool_args={"origin": "SYD", "destination": "LAX", "date": "2026-06-15"},
    tool_result=[{"flight": "QF11", "price": 1850}],
    sequence=0,
    previous_receipt_hash=None,
    signing_key=signing_key,
    verify_key=verify_key,
)

r1 = make_receipt(
    tool_name="hold_seat",
    tool_args={"flight": "QF11", "seat": "14A", "hold_minutes": 30},
    tool_result={"hold_id": "H8472", "expires_at": "2026-06-15T15:00:00Z"},
    sequence=1,
    previous_receipt_hash=receipt_hash(r0),
    signing_key=signing_key,
    verify_key=verify_key,
)

r2 = make_receipt(
    tool_name="confirm_booking",
    tool_args={"hold_id": "H8472", "payment_token": "tok_redacted"},
    tool_result={"booking_ref": "CT-09182", "status": "confirmed"},
    sequence=2,
    previous_receipt_hash=receipt_hash(r1),
    signing_key=signing_key,
    verify_key=verify_key,
)

chain = [r0, r1, r2]
for i, r in enumerate(chain):
    print(f"Receipt {i}: tool={r['tool_name']}, prev={r['previous_receipt_hash']}")

Receipt 0: tool=lookup_flights, prev=None
Receipt 1: tool=hold_seat, prev=sha256:af8b9159b25728881ce6a7166429f4451b9fa869a337453d52228f12131192e7
Receipt 2: tool=confirm_booking, prev=sha256:b1634969232be232f53283251cc9307b443a296909d9e74c2b0101b7f4dc679e


In [12]:
def verify_chain(chain: list, trusted_public_key: str) -> list[dict]:
    """
    Verify a sequence of receipts:
      1. Each receipt's signature must verify.
      2. Each receipt must carry previous_receipt_hash: null for the genesis, and
         the previous receipt's hash for every later receipt.
      3. Each sequence number must be an integer equal to the receipt's zero-based
         position in the chain.
    Returns a list of per-receipt result dicts. A malformed entry fails its own
    checks; it does not stop the rest of the chain from being checked.
    """
    results = []
    for i, receipt in enumerate(chain):
        is_receipt = isinstance(receipt, dict)
        sig_ok = verify_receipt(receipt, trusted_public_key)

        # The link field must be present: .get() alone would read a missing field as None.
        has_link = is_receipt and "previous_receipt_hash" in receipt
        link = receipt.get("previous_receipt_hash") if is_receipt else None
        if i == 0:
            chain_ok = has_link and link is None
        else:
            try:
                chain_ok = has_link and link == receipt_hash(chain[i - 1])
            except Exception:
                chain_ok = False  # the previous entry could not be hashed

        # type() rather than isinstance(): bool is a subclass of int, and True == 1.
        sequence = receipt.get("sequence") if is_receipt else None
        seq_ok = type(sequence) is int and sequence == i
        shape_ok = is_receipt and isinstance(receipt.get("tool_name"), str)

        results.append({
            "index": i,
            "tool": str(receipt.get("tool_name", "?")) if is_receipt else "(not a receipt)",
            "signature_valid": sig_ok,
            "chain_link_valid": chain_ok,
            "sequence_valid": seq_ok,
            "overall_valid": sig_ok and chain_ok and seq_ok and shape_ok,
        })
    return results

for r in verify_chain(chain, public_key_b64):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}")

Receipt 0 (    lookup_flights): VALID
Receipt 1 (         hold_seat): VALID
Receipt 2 (   confirm_booking): VALID


Brechen Sie nun die Kette, indem Sie den mittleren Beleg manipulieren, und verifizieren Sie erneut. Der manipulierte Beleg besteht die Signaturprüfung nicht, UND der nächste Beleg besteht die Kettenverknüpfungsprüfung nicht (weil sein `previous_receipt_hash` nicht mehr mit dem Hash des veränderten mittleren Belegs übereinstimmt).


In [13]:
# Tamper with the middle receipt: change the hold duration to something
# more permissive than was actually authorized.
tampered_chain = [copy.deepcopy(r) for r in chain]
tampered_chain[1]["tool_args_hash"] = sha256_canonical(
    {"flight": "QF11", "seat": "14A", "hold_minutes": 9999}
)

for r in verify_chain(tampered_chain, public_key_b64):
    status = "VALID" if r["overall_valid"] else "INVALID"
    why = ""
    if not r["overall_valid"]:
        reasons = []
        if not r["signature_valid"]:
            reasons.append("signature")
        if not r["chain_link_valid"]:
            reasons.append("chain link")
        if not r["sequence_valid"]:
            reasons.append("sequence")
        why = " (failed: " + ", ".join(reasons) + ")"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}{why}")

Receipt 0 (    lookup_flights): VALID
Receipt 1 (         hold_seat): INVALID (failed: signature)
Receipt 2 (   confirm_booking): INVALID (failed: chain link)


Beleg 0 verifiziert weiterhin (er wurde nicht verändert und hat keinen Vorgänger, auf den er angewiesen ist). Beleg 1 schlägt bei der Signaturprüfung fehl, weil wir `tool_args_hash` geändert haben. Beleg 2 schlägt bei der Ketten-Verknüpfungsprüfung fehl, da sein `previous_receipt_hash` gegen den ursprünglichen (jetzt modifizierten) Beleg 1 berechnet wurde.

Selbst wenn ein Angreifer den modifizierten Beleg 1 neu signiert, schlägt die neue Signatur gegen den Schlüssel des Ausstellers fehl, es sei denn, er besitzt den privaten Schlüssel des Ausstellers (eine Neusignierung mit seinem eigenen Schlüssel schlägt fehl, wie Abschnitt 2 gezeigt hat), und die Ketten-Verknüpfungsabweichung im Beleg 2 würde die Manipulation weiterhin offenlegen. Um die Änderung zu verbergen, müsste der Angreifer jeden Beleg ab dem Änderungszeitpunkt neu signieren, was den Besitz des privaten Schlüssels des Ausstellers erfordert.


## Abschnitt 4: Verpacken eines Agenten-Toolaufrufs mit Belegsignatur

In einem echten Einsatz möchten Sie nicht, dass jeder Agentenautor daran denken muss, `make_receipt` aufzurufen. Sie möchten, dass die Belegsignatur bei jedem Toolaufruf automatisch erfolgt.

Hier ist das einfachste Muster: eine Wrapper-Klasse, die jede aufrufbare Tool-Funktion entgegennimmt und eine Beleg-ausgebende Version davon zurückgibt. Dies passt sich jedem Agenten-Framework an, einschließlich des Microsoft Agent Frameworks (`agent_framework.foundry`).

Wenn Sie kein Microsoft Foundry-Projekt eingerichtet haben, zeigt das lokale Mock unten dennoch das Muster.


In [14]:
class ReceiptedTool:
    """
    Wraps a tool function so every invocation produces a signed receipt.
    Receipts are appended to a chain held by this object.

    Accepts both positional and keyword arguments. The receipt's
    tool_args field records args (as a list) and kwargs (as a dict)
    so the canonical hash binds to whichever the caller supplied.
    """

    def __init__(self, name: str, fn, signing_key, verify_key, agent_id: str, policy_id: str):
        self.name = name
        self.fn = fn
        self.signing_key = signing_key
        self.verify_key = verify_key
        self.agent_id = agent_id
        self.policy_id = policy_id
        self.receipts: list = []

    def __call__(self, *args, **kwargs):
        result = self.fn(*args, **kwargs)
        previous_hash = receipt_hash(self.receipts[-1]) if self.receipts else None
        receipt = make_receipt(
            tool_name=self.name,
            tool_args={"args": list(args), "kwargs": kwargs},
            tool_result=result,
            sequence=len(self.receipts),
            previous_receipt_hash=previous_hash,
            signing_key=self.signing_key,
            verify_key=self.verify_key,
            agent_id=self.agent_id,
            policy_id=self.policy_id,
        )
        self.receipts.append(receipt)
        return result

In [15]:
# Example tool: a mock flight lookup. In a real Microsoft Agent Framework deployment,
# this would be a function passed to FoundryChatClient as a tool.
def mock_lookup_flights(origin: str, destination: str, departure_date: str) -> list:
    return [
        {"flight": "QF11", "price": 1850, "stops": 0},
        {"flight": "UA864", "price": 1620, "stops": 1},
    ]

# Wrap it with receipt signing.
receipted_lookup = ReceiptedTool(
    name="lookup_flights",
    fn=mock_lookup_flights,
    signing_key=signing_key,
    verify_key=verify_key,
    agent_id="contoso-travel-bot",
    policy_id="contoso-travel-policy-v3",
)

# Use the wrapped tool exactly like the original.
results_a = receipted_lookup(origin="SYD", destination="LAX", departure_date="2026-06-15")
results_b = receipted_lookup(origin="SYD", destination="NRT", departure_date="2026-07-02")
results_c = receipted_lookup(origin="MEL", destination="SIN", departure_date="2026-08-10")

print(f"Tool was called {len(receipted_lookup.receipts)} times.")
print(f"Each call produced a signed receipt linked to the previous one.")
print()

for r in verify_chain(receipted_lookup.receipts, public_key_b64):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']}): {status}")


Tool was called 3 times.
Each call produced a signed receipt linked to the previous one.

Receipt 0 (lookup_flights): VALID
Receipt 1 (lookup_flights): VALID
Receipt 2 (lookup_flights): VALID


### Integration mit dem Microsoft Agent Framework

Der obenstehende `ReceiptedTool`-Wrapper ist framework-unabhängig. Um ihn in einem mit dem Microsoft Agent Framework erstellten Agenten zu verwenden, registrieren Sie die umwickelte Funktion als ein Tool. Eine Skizze (Sie würden den Mock durch eine echte Microsoft Foundry Tool-Registrierung ersetzen):

```python
# Pseudocode showing the integration shape.
# import os
# from agent_framework.foundry import FoundryChatClient
# from azure.identity import AzureCliCredential
#
# provider = FoundryChatClient(
#     project_endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
#     model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
#     credential=AzureCliCredential(),
# )
# agent = provider.as_agent(
#     instructions="You are a Contoso Travel agent ...",
#     tools=[receipted_lookup],   # the wrapped tool, not the raw function
# )
# response = agent.run("Find flights from Sydney to Los Angeles in June.")
#
# # After the run, every tool call the agent made has a signed receipt:
# audit_chain = receipted_lookup.receipts
```

Das Agent-Framework muss nichts über Belege wissen. Die Belegsignierung ist um das Tool herum eingebunden, nicht fest im Framework verankert. So fügen Sie bestehendem Agenten-Code Herkunftsinformationen hinzu, ohne den Agenten umzuschreiben.


## Rückblick und Stretch-Herausforderung

Du hast:

- Ein Ed25519-Schlüsselpaar generiert.
- Eine Quittung für einen Agenten-Tool-Aufruf erstellt und signiert.
- Die Quittung offline verifiziert, nur mit dem öffentlichen Schlüssel des Ausstellers, der unabhängig von der Quittung bezogen wurde.
- Eine Quittung manipuliert und einen Verifizierungsfehler beobachtet.
- Eine hash-verkettete Folge von drei Quittungen erstellt.
- Die Mitte der Kette manipuliert und sowohl einen Signaturfehler als auch einen Fehler bei der Verkettung beobachtet.
- Eine Tool-Funktion mit automatischer Quittierungsunterzeichnung umhüllt.

**Stretch-Herausforderung.** Erweitere das Quittungsschema um ein `request_id` Feld (eine UUID für verteiltes Tracing). Aktualisiere `make_receipt`, um es einzubeziehen, und bestätige, dass die Quittungen weiterhin Ende-zu-Ende verifizierbar sind. Verändere dann das Feld nach der Signierung und bestätige, dass die Verifizierung fehlschlägt. Dies zwingt dich, zu verinnerlichen, wie jedes Byte der kanonischen Kodierung zur Signatur beiträgt.

**Wichtige Grenze.** Unter einem fixierten Schlüssel- und Kettenkontext beweisen Quittungen Signaturzuordnung, Inhaltsintegrität und eine Verknüpfung zum angegebenen Vorgänger. Sie beweisen NICHT Ausführung, Quellwahrheit, vertrauenswürdige Zeit, Vollständigkeit, Politikevaluierung oder Korrektheit. Quittungen sind eine Grundlage. Governance ist das System, das du darauf aufbaust.

Lies die Lektion-README nochmals mit dieser Grenze im Hinterkopf. Der häufigste Fehler von Teams mit Quittungen ist anzunehmen, „wir haben Quittungen“ bedeutet „wir sind governert“. Das tut es nicht. Quittungen machen Agentenverhalten auditierbar. Sie machen es nicht korrekt.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Haftungsausschluss**:
Dieses Dokument wurde mit dem KI-Übersetzungsdienst [Co-op Translator](https://github.com/Azure/co-op-translator) übersetzt. Obwohl wir uns um Genauigkeit bemühen, beachten Sie bitte, dass automatisierte Übersetzungen Fehler oder Ungenauigkeiten enthalten können. Das Originaldokument in seiner Ursprungssprache gilt als maßgebliche Quelle. Bei kritischen Informationen wird eine professionelle menschliche Übersetzung empfohlen. Wir übernehmen keine Haftung für Missverständnisse oder Fehlinterpretationen, die aus der Verwendung dieser Übersetzung entstehen.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
